In [ ]:
# Resolve defaults from this project's root, independently of the working directory.
from pathlib import Path as _KSPath
import sys as _ks_sys
from functools import partial as _ks_partial
_ks_start = _KSPath.cwd()
_ks_root = next((p for p in (_ks_start, *_ks_start.parents) if (p / 'kinspec_paths.py').is_file()), None)
if _ks_root is None:
    raise RuntimeError('Cannot locate the KinSPEC project root containing kinspec_paths.py. Start the notebook kernel from the project directory or a subdirectory.')
_ks_sys.path.insert(0, str(_ks_root))
from kinspec_paths import ROOT as KINSPEC_ROOT, project_path as _ks_project_path
_kp = _ks_partial(_ks_project_path, script_dir='model_training')
import os as _ks_os
_ks_os.chdir(KINSPEC_ROOT / 'model_training')



In [ ]:
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.metrics import precision_recall_curve,log_loss,accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,matthews_corrcoef,roc_auc_score
import optuna
import gc
from scipy.stats import norm
from collections import Counter
from scipy.stats import norm
from functools import partial
from sklearn.model_selection import StratifiedKFold

In [ ]:
# Bin bioactivity values into binary labels
def bio_cal(bioactivity):
    # Clip PKIS1 values because some entries are above 100 or below 0
    bioactivity = bioactivity.clip(0,100)
    # Binning rule: define fixed intervals and their corresponding labels
    bins = [-np.inf, 50, np.inf]  # Intervals used to classify the normalized bioactivity values
    labels = [0, 1]  # Binary labels for activity strength
    
    final_bioactivity = pd.cut(
    bioactivity, 
    bins=bins, 
    labels=labels, 
    include_lowest=True
    )
    return final_bioactivity

In [ ]:
# import training data
warm_train = pd.read_csv(_kp('dataset/train_ready2use_merged_4datasets_deduplicated_with_smiles.csv'))

# import mol LSAE embedded features
warm_train_mol_fp = pd.read_csv(_kp('dataset/KinSPEC_training_pool_LSAE_mol_latent_top95_features_v4.csv'))

# import KLIFS seq from training data
warm_klifs = warm_train['KLIFS']

# import bioactivity classification from data
warm_bio = warm_train['unified_bio_label'] 

# mapping KLIFS to LSAE embedded KLIFS feature
KLIFS_sae_top = pd.read_csv(_kp('feature_embedding/Kinase_test_MCA/MCA_klifs_482_200D_features.csv'),skiprows=1, header=None).values
KLIFS_file = pd.read_csv(_kp('feature_embedding/Kinase_test_MCA/human_482_typical_kinase_1955bit.csv'))

In [ ]:
def klifs_map(df,KLIFS_file,KLIFS_latent): # df is the dataset to map; KLIFS_file contains all KLIFS entries; KLIFS_latent contains the corresponding reduced KLIFS features
    # Read KLIFS sequences from the input dataset
    read_klifs = df['KLIFS'].astype(str).str.strip()
    # Read KLIFS sequences from the KLIFS reference dataset
    KLIFS = KLIFS_file['KLIFS'].astype(str).str.strip()
    # Map each KLIFS sequence to its reduced latent representation
    keys = KLIFS
    values = KLIFS_latent
    KLIFS_dict = dict(zip(keys,values))
    # Perform the mapping
    kinase_result = [KLIFS_dict.get(x.lstrip(),None) for x in read_klifs]
    return kinase_result

In [ ]:
# Concatenate molecular latent features and kinase latent features
warm_train_kinase_fp = pd.DataFrame(klifs_map(warm_train,KLIFS_file,KLIFS_sae_top))
warm_train_input_feature = pd.concat([warm_train_mol_fp,warm_train_kinase_fp],axis=1)

In [ ]:
# check shape
print(warm_train.shape)
print(warm_train_mol_fp.shape)
print(warm_train_kinase_fp.shape)
print(warm_train_input_feature.shape)

In [ ]:
# Delete intermediate variables to reduce memory usage
del warm_train_mol_fp,warm_train,warm_klifs
# Run garbage collection
gc.collect()

In [ ]:
def dataset(input_feature, bioactivity):
    # Process feature labels
    input_feature.columns = range(input_feature.shape[1])
    # Safety check
    assert len(input_feature) == len(bioactivity), "The number of feature rows does not match the number of labels!"
    # Build the DMatrix object
    Xy = xgb.DMatrix(input_feature, label=bioactivity)
    return Xy,input_feature,bioactivity


In [ ]:
Xy_train,X_train,y_train= dataset(warm_train_input_feature,warm_bio)

In [ ]:
# set bedroc
def calculate_bedroc(y_true, y_score, bedroc_alpha):
    # y_true: true binary labels
    y_true  = np.asarray(y_true)
    # y_score: predicted scores
    y_score = np.asarray(y_score)
    N = len(y_true)
    n_act = np.sum(y_true == 1)
    if n_act == 0:
        return 0.0                    # Return 0 when there are no active compounds

    # 1. Sort predictions and extract the ranks of active compounds
    order  = np.argsort(y_score)[::-1]
    ranks  = np.arange(1, N+1)[y_true[order] == 1]      # 1-based
    x      = (ranks - 0.5) / N                          # Centered rank

    # 2. Calculate RIE
    C      = (1.0 - np.exp(-bedroc_alpha)) / bedroc_alpha
    rie    = np.mean(np.exp(-bedroc_alpha * x)) / C

    # 3. Calculate the theoretical limits
    x_max  = (np.arange(1, n_act+1) - 0.5) / N          # Best case
    x_min  = (np.arange(N-n_act+1, N+1) - 0.5) / N      # Worst case
    rie_max = np.mean(np.exp(-bedroc_alpha * x_max)) / C
    rie_min = np.mean(np.exp(-bedroc_alpha * x_min)) / C

    # 4. Normalize the score
    return (rie - rie_min) / (rie_max - rie_min)

In [ ]:
# Calculate the enrichment factor
def calculate_ef(y_true, y_score, percentages=[0.5, 1.0, 2.0, 3.0,4.0, 5.0]):

    # y_true: true binary labels
    y_true = np.asarray(y_true)
    # y_score: predicted scores
    y_score = np.asarray(y_score)
    
    # Check the inputs
    if len(y_true) != len(y_score):
        raise ValueError("y_true and y_score must have the same length")
    
    N = len(y_true)  # Total number of compounds
    n_active = np.sum(y_true == 1)  # Total number of active compounds
    
    if n_active == 0:
        print("Warning: no active compounds were found; returning EF = 0")
        return {f"EF_{p}%": 0.0 for p in percentages}
    
    # Sort compounds by predicted score in descending order
    sorted_indices = np.argsort(y_score)[::-1]
    sorted_labels = y_true[sorted_indices]
    
    results = {}
    
    for percentage in percentages:
        # Calculate the number of compounds corresponding to the top x%
        n_selected = max(1, int(N * percentage / 100))  # Select at least one compound
        
        # Number of active compounds in the top x%
        n_active_in_selected = np.sum(sorted_labels[:n_selected])
        
        # Calculate the enrichment factor
        # EF = active-compound fraction in the top x% / active-compound fraction in the full dataset
        ef_value = (n_active_in_selected / n_selected) / (n_active / N)
        
        results[f"EF_{percentage}%"] = ef_value
        
        # Print detailed information
        print(f"EF_{percentage}%:")
        print(f"  - Selected compounds: {n_selected}/{N} ({percentage}%)")
        print(f"  - Active compounds among selected compounds: {n_active_in_selected}")
        print(f"  - Active-compound fraction: {n_active_in_selected/n_selected:.4f}")
        print(f"  - Overall active-compound fraction: {n_active/N:.4f}")
        print(f"  - EF value: {ef_value:.4f}")
        print()
    
    return results

In [ ]:
def objective(trial, X, y, n_folds, bedroc_alpha):

    # ---------- 1. Search space ----------
    params = {
        "tree_method": "hist",     # GPU histogram algorithm
        "device": 'cuda',  # GPU prediction/training device
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "random_state": 42,

        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":     trial.suggest_int("max_depth", 5, 10),
        "subsample":     trial.suggest_float("subsample", 0.3, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 0.8),
        "lambda": trial.suggest_float("lambda", 1e-3, 1e3, log=True),
        "alpha":  trial.suggest_float("alpha", 1e-3, 300, log=True),
        "gamma":  trial.suggest_float("gamma", 0, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 12),
        "scale_pos_weight": 14
    }

    # Stratified cross-validation to preserve the positive-class ratio in each fold
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    all_y_true = []
    all_y_pred = []
    best_iters = []  # Store the best iteration number from each fold
    fold_aucs = [] # record for mediumpruner

    for fold_id, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        dtrain = Xy_train.slice(train_idx)
        dval = Xy_train.slice(valid_idx)

        # ---------- 4. Model training with early stopping ----------
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=10000,
            evals=[(dval, "eval")],
            early_stopping_rounds=50,
            verbose_eval=False
        )
        best_iters.append(model.best_iteration + 1)  # Record the best iteration number; XGBoost uses 0-based indexing, so add 1

        # Predict probabilities
        y_pred = model.predict(dval, iteration_range=(0, model.best_iteration + 1))

        # Collect the results
        all_y_true.append(y[valid_idx])
        all_y_pred.append(y_pred)

        # report per fold for pruner
        fold_aucs.append(roc_auc_score(y[valid_idx], y_pred))
        trial.report(float(np.mean(fold_aucs)), step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()

         # Calculate the average best iteration number
    avg_best_iter = int(np.round(np.mean(best_iters)))
    trial.set_user_attr('avg_best_iter', avg_best_iter)  # Save it as a user attribute

        # Concatenate predictions and labels
    y_true_concat = np.concatenate(all_y_true)
    y_pred_concat = np.concatenate(all_y_pred)

    # ---------- 5. Final evaluation ----------

    # Calculate BEDROC
    bedroc = calculate_bedroc(y_true_concat, y_pred_concat, bedroc_alpha=bedroc_alpha)
    auc = roc_auc_score(y_true_concat, y_pred_concat)
        # Save both metrics
    trial.set_user_attr("roc_auc", auc)
    trial.set_user_attr("bedroc", bedroc)
    
    print(f"[Validation] BEDROC alpha = {bedroc_alpha}")
    print(f"the best iteration number is {avg_best_iter}")
    return auc

In [ ]:
# Run Optuna Bayesian optimization
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2), # prune the trial if not good
    sampler=optuna.samplers.TPESampler(seed=42) # random seed
    )
study.optimize(partial(objective,X = X_train, y=y_train, n_folds=5,bedroc_alpha=80.5), n_trials=200,show_progress_bar=True)
best_params = study.best_params
best_iter = study.best_trial.user_attrs['avg_best_iter']
print("Best hyperparameters:", best_params)
print("Best iter:", best_iter)

In [ ]:
final_params = {
    'objective': 'binary:logistic',  # Binary classification task
    'eval_metric': 'auc',       # Evaluation metric
    'tree_method': 'hist',
    'device': 'cuda',
    'scale_pos_weight':14, # balance active/inactive
    **best_params
}
final_model = xgb.train(final_params, Xy_train, num_boost_round=best_iter)
final_model.save_model(_kp('model/KinSPEC/KinSPEC_full_training_pool_MCA_200D_v4.json'))


In [ ]:
del Xy_train,X_train,y_train,warm_train_input_feature,warm_bio
gc.collect()

In [ ]:
# Define a function to test the trained model
def run_the_model_full(Xy,y,kinase_name):

    # Load the model
    final_model = xgb.Booster()
    final_model.load_model(_kp('model/KinSPEC/KinSPEC_full_training_pool_MCA_200D_v4.json'))
    y_pred_prob = final_model.predict(Xy)  # Predicted probabilities

    # Automatically select the optimal threshold
    precisions, recalls, thresholds = precision_recall_curve(y, y_pred_prob)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    optimal_idx = np.argmax(f1_scores)
    y_pred = (y_pred_prob > thresholds[optimal_idx])

    # Calculate common evaluation metrics
    auc = roc_auc_score(y, y_pred_prob)
    auc_var = np.var(y_pred_prob) / len(y_pred_prob)
    z = norm.ppf(1 - (1 - 0.95) / 2)  # Calculate the z value
    lower_bound = auc - z * np.sqrt(auc_var)
    upper_bound = auc + z * np.sqrt(auc_var)

    acc = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    mcc = matthews_corrcoef(y, y_pred)
    logloss = log_loss(y, y_pred_prob)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) != 0 else 0.0

     # Calculate confidence intervals by bootstrapping
    n_bootstraps = 1000
    bootstrapped_aucs = []

    # Convert y and y_pred_prob to NumPy arrays
    y = np.array(y)
    y_pred_prob = np.array(y_pred_prob)

    rng = np.random.RandomState(42)
    for _ in range(n_bootstraps):
        indices = rng.choice(len(y), size=len(y), replace=True)
        auc_bootstrap = roc_auc_score(y[indices], y_pred_prob[indices])
        bootstrapped_aucs.append(auc_bootstrap)
    ci_lower = np.percentile(bootstrapped_aucs, 2.5)
    ci_upper = np.percentile(bootstrapped_aucs, 97.5)
    print(f"AUC: {auc:.3f} (95% CI: [{ci_lower:.3f}, {ci_upper:.3f}])")

    # Calculate BEDROC
    bedroc_score = calculate_bedroc(y, y_pred_prob, bedroc_alpha=80.5)
    ef_score = calculate_ef(y, y_pred_prob)
    # Print evaluation results
    print(f"Best threshold: {thresholds[optimal_idx]:.4f}")
    print(f"AUC: {auc:.4f} (95% CI: {lower_bound:.3f}, {upper_bound:.3f})")
    print(f"bedroc: {bedroc_score:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision(PPV): {precision:.4f}")
    print(f"Recall(Sensitivity): {recall:.4f}")
    print(f"F1 score: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"NPV: {npv:.4f}")
    print(f"MCC: {mcc:.4f}")
    print(f"Log Loss: {logloss:.4f}")
    print(confusion_matrix(y, y_pred))
    print(f"True Negative = {tn}")
    print(f"True Positive = {tp}")
    print(f"False Negative = {fn}")
    print(f"False Positive = {fp}")
    print("Class distribution:", Counter(y))
    print("Positive-class fraction:", np.mean(y))

    if kinase_name is not None:
        df_output = pd.DataFrame({
            "Best threshold": f"{thresholds[optimal_idx]:.4f}",
            "AUC": f"{auc:.4f}",
            "AUC 95% CI lower_bound": f"{lower_bound:.3f}",
            "AUC 95% CI upper_bound": f"{upper_bound:.3f}",
            "bedroc": f"{bedroc_score:.4f}",
            "Accuracy": f"{acc:.4f}",
            "Precision(PPV)": f"{precision:.4f}",
            "Recall(Sensitivity)": f"{recall:.4f}",
            "F1 score": f"{f1:.4f}",
            "Specificity": f"{specificity:.4f}",
            "NPV": f"{npv:.4f}",
            "MCC": f"{mcc:.4f}",
            "Log Loss": f"{logloss:.4f}",
            "True Negative": f"{tn}",
            "True Positive": f"{tp}",
            "False Negative": f"{fn}",
            "False Positive": f"{fp}"
        },index=[kinase_name])
        return bedroc_score,df_output,ef_score
    else:
        return bedroc_score

In [ ]:
# vsds test dataset
vsds_mol_fp = pd.read_csv(_kp('dataset/VSDS_truedecoy_LSAE_latent_top95_features_v4.csv'),skiprows=1,header=None)

vsds = pd.read_csv(_kp('dataset/VSDS_truedecoy_integrated_data_with_protein_seq.csv'))

vsds_klifs = vsds['KLIFS']

vsds_bio = vsds['molecule_label']

vsds_kinase_fp = pd.DataFrame(klifs_map(vsds,KLIFS_file,KLIFS_sae_top))
vsds_input_feature = pd.concat([vsds_mol_fp,vsds_kinase_fp],axis=1)

vsds_input_feature.columns = range(vsds_input_feature.shape[1])

print(vsds_mol_fp.shape)
print(vsds_kinase_fp.shape)

vsds_Xy = xgb.DMatrix(vsds_input_feature,label=vsds_bio)

In [ ]:
print('evaluation metrics on VSDS test set:')
vsds_test = run_the_model_full(vsds_Xy,vsds_bio,None)